In [ ]:
import pandas as pd

df = pd.read_csv("DataSet1-raw_listings.csv")

print(f"{df.shape[0]}/2040 listings and {df.shape[1]}/46 attributes loaded successfully.")

2040/2040 listings and 46/46 attributes loaded successfully.


In [37]:
print(df.head())

                                id                vin  \
0  1B7HC16YXSS331012-aec807ef-8d90  1B7HC16YXSS331012   
1  1C3EU4532SF629087-74640eef-b47f  1C3EU4532SF629087   
2  WDBGA51E8SA237429-34a36c54-9fc6  WDBGA51E8SA237429   
3  1G6KD52B1SU253285-51611ab2-6365  1G6KD52B1SU253285   
4  1G4HP52L5SH550801-d39fcf03-0ee6  1G4HP52L5SH550801   

                                 heading  price   miles  msrp data_source  \
0           1995 Dodge Ram 1500 LT Truck   5548  160280  5548          mc   
1  1995 Chrysler Lebaron GTC Convertible   5950   68284  5950          mc   
2              1995 MERCEDES-BENZ SCLASS   5988  117446  5988          mc   
3        1995 Cadillac Deville 4DR SEDAN   5909  109633  5909          mc   
4               1995 Buick LeSabre Sedan   4799  133432  4799          mc   

                                             vdp_url carfax_1_owner  \
0  https://www.bobthomasfordnorth.net/used/Dodge/...          False   
1  https://www.millermotorschryslerdodgejeep.com/...

In [38]:
print(df.isnull().sum())

id                              0
vin                             0
heading                         2
price                           0
miles                           0
msrp                            0
data_source                     0
vdp_url                         0
carfax_1_owner                 87
carfax_clean_title             89
exterior_color                153
interior_color                542
base_int_color                603
base_ext_color                166
dom                             0
dom_180                         0
dom_active                      0
dos_active                      0
seller_type                     0
inventory_type                  0
stock_no                      288
last_seen_at                    0
last_seen_at_date               0
scraped_at                      0
scraped_at_date                 0
first_seen_at                   0
first_seen_at_date              0
first_seen_at_source            0
first_seen_at_source_date       0
first_seen_at_

In [39]:
df = df.drop_duplicates(subset='vin')

In [40]:
heading_fixes = {
    'Chevroletsilverado': '2004 Chevrolet Silverado',
    'Dodgestratus': 'Dodge Stratus',
    'Fordtaurus': 'Ford Taurus',
    'Hyundaisanta': 'Hyundai Santa Fe',
    'Lexussc': 'Lexus SC 430',
    'Pontiacvibe': 'Pontiac Vibe',
}

for wrong, correct in heading_fixes.items():
    mask = df['heading'].str.contains(wrong, case=False, na=False)
    df.loc[mask, 'heading'] = df.loc[mask, 'heading'].str.replace(wrong, correct, case=False, regex=False)

In [41]:
df['heading'] = df['heading'].str.replace('MERCEDES-BENZ', 'Mercedes-Benz', regex=False)
df['heading'] = df['heading'].str.replace('Mercedes ', 'Mercedes-Benz ', regex=False)

In [42]:
for wrong in heading_fixes.keys():
    remaining = df['heading'].str.contains(wrong, case=False, na=False).sum()
    print(f"{wrong}: {remaining} remaining")

Chevroletsilverado: 0 remaining
Dodgestratus: 0 remaining
Fordtaurus: 0 remaining
Hyundaisanta: 0 remaining
Lexussc: 0 remaining
Pontiacvibe: 0 remaining


In [43]:
dash_fixes = {
    '-C-O-B-A-L-T-': 'Cobalt',
    '-G-R-A-N-D-': 'Grand Marquis',
    '-G-6-': 'G6',
    '-G-L-I-': 'Gli',
    '-E-2-5-0-': 'E-250',
    '-A-C-C-O-R-D-': 'Accord',
    '-Z-3-': 'Z3',
    '-Z-4-': 'Z4',
    '-C-3-0-': 'C30',
    '-V-5-0-': 'V50',
}

for wrong, correct in dash_fixes.items():
    df['heading'] = df['heading'].str.replace(wrong, correct, regex=False)

for wrong in dash_fixes.keys():
    remaining = df['heading'].str.contains(wrong, na=False, regex=False).sum()
    print(f"{wrong}: {remaining} remaining")

-C-O-B-A-L-T-: 0 remaining
-G-R-A-N-D-: 0 remaining
-G-6-: 0 remaining
-G-L-I-: 0 remaining
-E-2-5-0-: 0 remaining
-A-C-C-O-R-D-: 0 remaining
-Z-3-: 0 remaining
-Z-4-: 0 remaining
-C-3-0-: 0 remaining
-V-5-0-: 0 remaining


In [44]:
no_year_mask = ~df['heading'].str.contains(r'(?:19|20)\d{2}', regex=True, na=True)
print(f"Dropping {no_year_mask.sum()} rows with no year in heading")
df = df[~no_year_mask]
print(f"Remaining rows: {len(df)}")

Dropping 32 rows with no year in heading
Remaining rows: 2008


In [ ]:
import re

df['year'] = df['heading'].str.extract(r'((?:19|20)\d{2})')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)

df['make'] = df['heading'].str.extract(r'(?:19|20)\d{2}\s+(\w+)')
df['model'] = df['heading'].str.extract(r'(?:19|20)\d{2}\s+\w+\s+([\w-]+)')

df['make'] = df['make'].str.title().str.strip()
df['model'] = df['model'].str.title().str.strip()

df = df.dropna(subset=['make', 'model'])

print(f"Rows after extraction: {len(df)}")
print(df[['heading', 'year', 'make', 'model']].head(20))

Rows after extraction: 1932
                                              heading  year       make  \
0                        1995 Dodge Ram 1500 LT Truck  1995      Dodge   
1               1995 Chrysler Lebaron GTC Convertible  1995   Chrysler   
3                     1995 Cadillac Deville 4DR SEDAN  1995   Cadillac   
4                            1995 Buick LeSabre Sedan  1995      Buick   
5           1995 Dodge Ram 2500 Reg Cab 135 WB HD 4WD  1995      Dodge   
6                                      1995 JAGUAR XJ  1995     Jaguar   
7              1995 Chevrolet Camaro Base Convertible  1995  Chevrolet   
8                1995 Ford F-150 XL Truck Regular Cab  1995       Ford   
9                          1996 Toyota Camry LE Sedan  1996     Toyota   
10                    1996 Dodge Dakota Classic Truck  1996      Dodge   
11                  1996 Lincoln Mark VIII Base Coupe  1996    Lincoln   
12                                1996 MERCURY COUGAR  1996    Mercury   
13    1996

In [ ]:
model_mapping = {
    'F150': 'F-150',
    'F250': 'F-250',
    'F350': 'F-350',
    'F-350Sd': 'F-350',
    'F450': 'F-450',
    'E150': 'E-150',
    
    'Crv': 'Cr-V',
    
    'Silverado-1500': 'Silverado',

    'Grand-Caravan': 'Grand Caravan',
    'Grand-Am': 'Grand Am',
    
    'Grand-Marquis': 'Grand Marquis',
    
    'Santa-Fe': 'Santa Fe',
    
    'Cx9': 'Cx-9',
    'Mazda3-Hatchback': 'Mazda3',
    '6': 'Mazda6',
    
    '93': '9-3',
    
    '5Series': '5-Series',
    
    'Xtype': 'X-Type',
    'Xj-Series': 'Xj',

    'Impreza-Outback-Sport': 'Impreza',
    
    'New': 'Beetle',
    
    'Es-350': 'Es',
    
    'Xl-7': 'Xl7',
}

df['model'] = df['model'].replace(model_mapping)

print(df['model'].value_counts())

model
Camry               72
F-150               71
Accord              64
Grand               46
Civic               43
                    ..
Traverse             1
Gli                  1
E-350-Super-Duty     1
A5                   1
M                    1
Name: count, Length: 314, dtype: int64


In [ ]:
truncation_fixes = {
    ('Ford', 'Crown'): 'Crown Victoria',
    ('Ford', 'Five'): 'Five Hundred',
    ('Ford', 'Super'): 'Super Duty',
    ('Chrysler', 'Town'): 'Town And Country',
    ('Chrysler', 'Pt'): 'Pt Cruiser',
    ('Lincoln', 'Town'): 'Town Car',
    ('Lincoln', 'Mark'): 'Mark Viii',
    ('Mini', 'John'): 'John Cooper Works',
    ('Jeep', 'Grand'): 'Grand Cherokee',
    ('Dodge', 'Grand'): 'Grand Caravan',
    ('Pontiac', 'Grand'): 'Grand Am',
    ('Mercury', 'Grand'): 'Grand Marquis',
    ('Suzuki', 'Grand'): 'Grand Vitara',
    ('Land', 'Rover'): 'Range Rover',
}

for (make, wrong_model), correct_model in truncation_fixes.items():
    mask = (df['make'] == make) & (df['model'] == wrong_model)
    print(f"{make} {wrong_model}: {mask.sum()} rows → {correct_model}")
    df.loc[mask, 'model'] = correct_model

df.loc[df['make'] == 'Land', 'make'] = 'Land Rover'

Ford Crown: 3 rows → Crown Victoria
Ford Five: 1 rows → Five Hundred
Ford Super: 1 rows → Super Duty
Chrysler Town: 12 rows → Town And Country
Chrysler Pt: 20 rows → Pt Cruiser
Lincoln Town: 13 rows → Town Car
Lincoln Mark: 1 rows → Mark Viii
Mini John: 1 rows → John Cooper Works
Jeep Grand: 17 rows → Grand Cherokee
Dodge Grand: 9 rows → Grand Caravan
Pontiac Grand: 5 rows → Grand Am
Mercury Grand: 13 rows → Grand Marquis
Suzuki Grand: 2 rows → Grand Vitara
Land Rover: 7 rows → Range Rover


In [ ]:
df.loc[df['make'] == 'Mercedes', 'make'] = 'Mercedes-Benz'

df.loc[df['make'] == 'Mazda3', 'model'] = 'Mazda3'
df.loc[df['make'] == 'Mazda3', 'make'] = 'Mazda'

df = df[df['make'] != 'Sterling']

df = df[df['make'] != 'Fordf']

print(f"Rows remaining: {len(df)}")
print(df['make'].value_counts())

Rows remaining: 1931
make
Ford          289
Toyota        200
Chevrolet     168
Honda         165
Nissan        107
Dodge         102
Chrysler       73
Hyundai        70
Jeep           66
Bmw            58
Buick          51
Subaru         50
Volkswagen     45
Cadillac       44
Lexus          44
Gmc            39
Lincoln        30
Mercury        30
Acura          30
Pontiac        29
Mazda          28
Volvo          28
Kia            27
Saturn         21
Infiniti       21
Mitsubishi     18
Mini           17
Audi           16
Scion          16
Jaguar         13
Oldsmobile      7
Land Rover      7
Suzuki          5
Hummer          5
Smart           5
Saab            3
Isuzu           2
Porsche         1
2004            1
Name: count, dtype: int64


In [ ]:
df['make'] = df['make'].str.title().str.strip()
df['model'] = df['model'].str.title().str.strip()

df = df.drop_duplicates(subset='vin')

# Final check
print(f"Final row count: {len(df)}")
print(f"Unique makes: {df['make'].nunique()}")
print(f"Unique models: {df['model'].nunique()}")
print(f"Unique make/model combinations: {df.groupby(['make', 'model']).ngroups}")
print(df[['make', 'model']].drop_duplicates().sort_values(['make', 'model']).to_string())

Final row count: 1931
Unique makes: 39
Unique models: 314
Unique make/model combinations: 315
            make              model
151         2004          Chevrolet
174        Acura                Mdx
1201       Acura                Rdx
723        Acura                 Rl
190        Acura                 Tl
611        Acura                Tsx
1605        Audi                 A3
254         Audi                 A4
1977        Audi                 A5
1016        Audi                 A6
439         Audi                 A8
665         Audi                 S4
247         Audi                 Tt
1449         Bmw               128I
153          Bmw                  3
119          Bmw               323I
462          Bmw               325I
372          Bmw              325Xi
1456         Bmw               328I
991          Bmw              328Xi
331          Bmw              330Ci
1116         Bmw               335I
885          Bmw                  5
1542         Bmw           5-Series
30    

In [ ]:
df = df[df['make'] != '2004']

bmw_drop = ['3', '5', 'M', '7', '528', 'Mazda6']
df = df[~((df['make'] == 'Bmw') & (df['model'].isin(bmw_drop)))]

df = df[~((df['make'] == 'Chrysler') & (df['model'] == '300'))]

df = df[~((df['make'] == 'Infiniti') & (df['model'] == 'G'))]

jaguar_drop = ['Xj', 'Xj8', 'Xk', 'Xk8', 'Xkr', 'Xtype', 'Xf']
df = df[~((df['make'] == 'Jaguar') & (df['model'].isin(jaguar_drop)))]

df = df[~((df['make'] == 'Mazda') & (df['model'] == 'Mazda'))]

df = df[~((df['make'] == 'Hyundai') & (df['model'] == 'Santa'))]

print(f"Rows remaining: {len(df)}")
print(f"Unique make/model combinations: {df.groupby(['make', 'model']).ngroups}")

Rows remaining: 1867
Unique make/model combinations: 298


In [ ]:
df.loc[(df['make'] == 'Chevrolet') & (df['model'] == 'Monte'), 'model'] = 'Monte Carlo'
df.loc[(df['make'] == 'Saturn') & (df['model'] == 'Lseries'), 'model'] = 'L-Series'
df.loc[(df['make'] == 'Ford') & (df['model'] == 'E-350-Super-Duty'), 'model'] = 'E-350 Super Duty'

df = df[~((df['make'] == 'Infiniti') & (df['model'] == 'Qx'))]
df = df[~((df['make'] == 'Bmw') & (df['model'] == '530Xi'))]
df = df[~((df['make'] == 'Gmc') & (df['model'] == 'B-Series'))]

print(f"Rows remaining: {len(df)}")
print(f"Unique make/model combinations: {df.groupby(['make', 'model']).ngroups}")

df.to_csv("DataSet2-cleaned_listings.csv", index=False)
print("Saved DataSet2-cleaned_listings.csv")

Rows remaining: 1864
Unique make/model combinations: 295
Saved cleaned_listings.csv


**Wierdly Enough, MarketCheckAPI has Make and Model as params in the API documentation. How do they make it work if I needed all this normalization?**